## Advanced Chunking

### Semantic Chunking
- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

e.g
- LangChain is a framework for building applications with LLMs.
- Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
- You can create chains, agents, memory, and retrievers.
- The Eiffel Tower is located in Paris.
- France is a popular tourist destination.
    - Like in the above example in a single document- There are two different meaning sentences, one related to Langchain and other is for Eiffle Tower
    - Using Recursive Text Spilliter will mix up the sentences and context we will get will not be logically related
    - So we can use this Advanced Chunking Technique

How it works:
- Create chunks based onn sentences/paragraphs( chunk_size, chunk_overlap, separators, recursive text splitter)
- Use sentence embeddings like from openai sentence transformers
- Compare the chunks with the scores(CoSine Simalarity)
- Define a threshold and according to that combine the embeddings that are more similar
- Idea is create a more contextually rich and logically separated

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

/home/divyanshu/Desktop/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

##Sample Text
text= """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8959.96it/s]


In [3]:
#Step 1. Split the text into sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

#Step 2. Generate embeddings for each sentence
embeddings = model.encode(sentences)
#Step 3. Initialize parameters for semantic chunking
similarity_threshold = 0.7  # Adjust this threshold based on your needs
chunks = []
current_chunk = [sentences[0]]

#Step 4. Iterate through the sentences and group them based on semantic similarity
for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i-1]],
        [embeddings[i]]
    )[0][0]                   
#[
#  [0.99258333] output of sim; so [0] =[0.99258333] and [0][0] = 0.99258333
#]
    if sim >= similarity_threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]
# Add the last chunk
if current_chunk:
    chunks.append(" ".join(current_chunk))

#Output the chunks
print("Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx+1}: {chunk}") 

Semantic Chunks:
Chunk 1: LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
Chunk 2: You can create chains, agents, memory, and retrievers.
Chunk 3: The Eiffel Tower is located in Paris.
Chunk 4: France is a popular tourist destination.


### RAG Pipeline Modular Coding


In [11]:
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.vectorstores import FAISS
from langchain.chat_models.base import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [5]:
##Custom Semantic Chunker With Threshold
class ThresholdSemanticChunker:
    def __init__(self, model_name="all-MiniLM-L6-v2", threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold= threshold
    def split(self,text):
        sentences=[s.strip() for s in text.split(".") if s.strip()]

        embeddings = self.model.encode(sentences)

        chunks=[]
        current_chunk=[sentences[0]]

        for i in range(1,len(sentences)):
            sim = cosine_similarity(
                [embeddings[i-1]],
                [embeddings[i]]
            )[0][0]
            if sim>=self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk=[sentences[i]]
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        return chunks
    
    def split_documents(self,docs):
        result=[]
        for doc in docs:
            chunks = self.split(doc.page_content)
            for chunk in chunks:
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result
            

In [6]:
##Sample Text
text= """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc =Document(page_content=text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [7]:
##Chunking
chunker= ThresholdSemanticChunker(threshold=0.7)
chunks= chunker.split_documents([doc])
chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11932.42it/s]


[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris'),
 Document(metadata={}, page_content='France is a popular tourist destination')]

In [ ]:
### Vector Store FAISS
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store= FAISS.from_documents(chunks, embeddings)

## as retriever
retriever= vector_store.as_retriever(search_type="similarity", search_kwargs={"k":2})
retriever.invoke

In [9]:
## Prompt Template
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

template = """
Answer the question based on the context below. If the answer is not contained within the text below, say "I don't know".
Context:{context}
Question: {question}
Answer:
"""

prompt = PromptTemplate(
    template=template,
    input_variables=[])

prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below. If the answer is not contained within the text below, say "I don\'t know".\nContext:{context}\nQuestion: {question}\nAnswer:\n')

In [13]:
llm = init_chat_model(
    model="qwen/qwen3-32b",
    model_provider="groq"
)

# LCEL
rag_chain=(
    {
        "context": lambda z: retriever.invoke(z["question"]),
        "question": lambda z: z["question"],
    }
    | prompt
    | llm
    | StrOutputParser()
)

query = rag_chain.invoke({"question": "What is Langchain?"})
print(query)

<think>
Okay, the user is asking "What is Langchain?" and I need to answer based on the provided context. Let me check the documents.

First document says: "LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone." So that's a direct definition. The second document mentions components like chains, agents, memory, and retrievers, which are part of what Langchain offers.

The answer should include that it's a framework for building apps with LLMs, and mention the modular abstractions and tools. Since both documents are part of the context, I should combine that info. I don't see any conflicting info, so the answer is covered in the first document's content. Need to present it clearly and concisely.
</think>

LangChain is a framework for building applications with Large Language Models (LLMs). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone, enabling funct

### Semantic chunker With Langchain

In [14]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_classic.document_loaders import TextLoader

In [19]:
## Load the documents
loader = TextLoader("./text_files/python_intro.txt")
docs = loader.load()

# Initialize Embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## Semantic CHunker
chunker = SemanticChunker(embeddings)

## SPlit the documents into semantic chunks
chunks = chunker.split_documents(docs)

# Result

for i,chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk.page_content}\n")   
    print(f"Metadata: {chunk.metadata}\n")

Chunk 1: Python is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world. Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support
Python is widely used in web development, data science, artificial intelligence, and automation. Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed.

Metadata: {'source': './text_files/python_intro.txt'}

Chunk 2: It focuses on developing computer programs
that can access data and use it to learn for themselves. Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications i

In [28]:
# Vector Store FAISS
vector_store= FAISS.from_documents(chunks, embeddings)

retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":2})
retriever.invoke("What is Python?")

[Document(id='05b63ee4-c89c-4bc3-910b-ce025471290a', metadata={'source': './text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world. Key Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\nPython is widely used in web development, data science, artificial intelligence, and automation. Machine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed.'),
 Document(id='ee9fa4e1-357b-47a5-a282-f9f90c661746', metadata={'source': './text_files/python_intro.txt'}, page_content='It focuses on developing computer programs\nthat can access data and use it to learn for themselves. Types of Machine Learning:\n1. Supervised

In [29]:
# prompt Template
template ="""
Answer the question based on the context below. If the answer is not contained within the text below, say "I don't know".
Context:{context}
Question: {question}
Answer:
"""

prompt = PromptTemplate(
    template=template,
    input_variables=[]
)

In [30]:
### LLM
llm_openai = init_chat_model(
    model="gpt-5-mini",
    model_provider="openai"
)

In [31]:
#LCEL
rag_chain =(
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],
    }
    | prompt
    | llm_openai    
    | StrOutputParser()
)

In [34]:
query = rag_chain.invoke({"question": "What is Python? and mention the chunk and its metadata "})
print(query)

Python is a high-level, interpreted programming language known for its simplicity and readability; it was created by Guido van Rossum and first released in 1991, and is widely used in web development, data science, AI, and automation.

Chunk (Document id: 05b63ee4-c89c-4bc3-910b-ce025471290a):
"Python is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world. Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support
Python is widely used in web development, data science, artificial intelligence, and automation. Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed."

Metadata:
{'source': './text_files/python_intro.txt'}
